# DQN: Donkey Kong (Default Hyperparameters)

**Algorithm:** DQN  
**Environment:** Donkey Kong (`ALE/DonkeyKong-v5`)  

In [ ]:
# Configuration
ALGO            = "dqn"
ENV_KEY         = "donkeykong"
TOTAL_TIMESTEPS = 2_000_000
CHECKPOINT_FREQ = 100_000
LEARNING_RATE   = None
RUN_NAME        = None

In [11]:
import os

import ale_py
import gymnasium as gym
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

gym.register_envs(ale_py)

ENV_IDS = {
    "qbert":      "QbertNoFrameskip-v4",
    "donkeykong": "ALE/DonkeyKong-v5",
}

run_id = f"{ALGO}_{ENV_KEY}" + (f"_{RUN_NAME}" if RUN_NAME else "")
checkpoint_dir = os.path.join("checkpoints", run_id)
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs("logs", exist_ok=True)

print(f"Run ID      : {run_id}")
print(f"Checkpoints : {checkpoint_dir}")

Run ID      : dqn_donkeykong
Checkpoints : checkpoints/dqn_donkeykong


In [12]:
n_envs = 8 if ALGO == "ppo" else 1
vec_env = make_atari_env(ENV_IDS[ENV_KEY], n_envs=n_envs, seed=0)
vec_env = VecFrameStack(vec_env, n_stack=4)
print(f"Environment : {ENV_IDS[ENV_KEY]}  |  n_envs: {n_envs}")

Environment : ALE/DonkeyKong-v5  |  n_envs: 1


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


## Fresh Training

Checkpoints saved automatically to `checkpoints/dqn_donkeykong/` every 100,000 steps.  
At 2M timesteps, expect checkpoints at 100K, 200K, … 2000K steps.

In [13]:
if ALGO == "dqn":
    kwargs = dict(
        verbose=1, device="cpu", tensorboard_log="logs",
        buffer_size=100_000, learning_starts=10_000,
        batch_size=32, train_freq=4, target_update_interval=1_000,
    )
    if LEARNING_RATE is not None:
        kwargs["learning_rate"] = LEARNING_RATE
    model = DQN("CnnPolicy", vec_env, **kwargs)
elif ALGO == "ppo":
    kwargs = dict(
        verbose=1, device="cpu", tensorboard_log="logs",
        n_steps=128, batch_size=256, n_epochs=4, ent_coef=0.01, clip_range=0.1,
    )
    if LEARNING_RATE is not None:
        kwargs["learning_rate"] = LEARNING_RATE
    model = PPO("CnnPolicy", vec_env, **kwargs)

print(f"Algorithm : {ALGO.upper()}")
print(f"Device    : {model.device}")

Using cpu device
Wrapping the env in a VecTransposeImage.
Algorithm : DQN
Device    : cpu


In [14]:
checkpoint_callback = CheckpointCallback(
    save_freq=max(CHECKPOINT_FREQ // n_envs, 1),
    save_path=checkpoint_dir,
    name_prefix="checkpoint",
    save_replay_buffer=False,
    save_vecnormalize=True,
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=checkpoint_callback,
    tb_log_name=run_id,
    log_interval=100,
    progress_bar=True,
    reset_num_timesteps=True,
)

final_path = os.path.join(checkpoint_dir, "final_model")
model.save(final_path)
print(f"\nTraining complete. Final model saved to: {final_path}.zip")

Logging to logs/dqn_donkeykong_1


Output()

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 743      |
|    ep_rew_mean      | 60       |
|    exploration_rate | 0.958    |
| time/               |          |
|    episodes         | 100      |
|    fps              | 736      |
|    time_elapsed     | 12       |
|    total_timesteps  | 8872     |
----------------------------------


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 756      |
|    ep_rew_mean      | 66       |
|    exploration_rate | 0.914    |
| time/               |          |
|    episodes         | 200      |
|    fps              | 123      |
|    time_elapsed     | 146      |
|    total_timesteps  | 18028    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0134   |
|    n_updates        | 2006     |
----------------------------------


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 753      |
|    ep_rew_mean      | 82       |
|    exploration_rate | 0.873    |
| time/               |          |
|    episodes         | 300      |
|    fps              | 91       |
|    time_elapsed     | 292      |
|    total_timesteps  | 26805    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00293  |
|    n_updates        | 4201     |
----------------------------------


...

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.76e+03 |
|    ep_rew_mean      | 1.62e+03 |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9300     |
|    fps              | 58       |
|    time_elapsed     | 33002    |
|    total_timesteps  | 1930693  |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.014    |
|    n_updates        | 480173   |
----------------------------------


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.87e+03 |
|    ep_rew_mean      | 1.75e+03 |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9400     |
|    fps              | 58       |
|    time_elapsed     | 33409    |
|    total_timesteps  | 1954388  |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00432  |
|    n_updates        | 486096   |
----------------------------------


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.92e+03 |
|    ep_rew_mean      | 1.79e+03 |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9500     |
|    fps              | 58       |
|    time_elapsed     | 33809    |
|    total_timesteps  | 1977750  |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00452  |
|    n_updates        | 491937   |
----------------------------------


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.84e+03 |
|    ep_rew_mean      | 1.71e+03 |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9600     |
|    fps              | 58       |
|    time_elapsed     | 34184    |
|    total_timesteps  | 1999661  |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00345  |
|    n_updates        | 497415   |
----------------------------------



Training complete. Final model saved to: checkpoints/dqn_donkeykong/final_model.zip
